# Fine-tune VLA-Adapter on 30 Hz IK three-block stacking

This notebook downloads the **1,000 scripted IK demonstrations / 862,766 frames** from Hugging Face and fully fine-tunes the Qwen2.5-0.5B VLA-Adapter model. The task is red onto green, then blue onto red. It converts the published LeRobot Parquet files directly to RLDS, preserving both embedded camera JPEGs and **native 30 Hz** timing. No MuJoCo rendering is needed.

Actions are six **absolute next-step joint targets** (radians) plus gripper opening (metres), already aligned to the observation at t to predict t + 1/30 s. They are not shifted again or converted to end-effector deltas. Proprio is six measured joints, one zero padding element, and the current gripper opening command. All seven action dimensions are normalized using training episodes only.

Before running, select **Runtime → Change runtime type** and pick an **A100 (recommended)**. Training uses bfloat16 and needs an Ampere-or-newer GPU. Full fine-tuning updates all model parameters; an L4 may need additional memory tuning. Allow at least **100 GiB free runtime disk** for the source dataset (about 22 GiB), RLDS, temporary conversion files, environment, and model; allow additional Drive space for checkpoints and the exported archive. Run this notebook from the top and leave `RESUME_RUN_ID` empty for the first IK run. Teleop/EEF checkpoints have a different action contract and cannot be resumed here.


In [ ]:
# User settings
DATASET_REPO = "FoxNerdSaysMoo/panthera-ik-three-block-stack-30hz"
DATASET_REVISION = "cc138c0ea9ec5332a7754ee1b1f5be4de75b4226"
CODE_REPO = "https://github.com/zebulontaylor/RobotArmTraining.git"
VLA_REPO = "https://github.com/OpenHelix-Team/VLA-Adapter.git"
VLA_COMMIT = "23fa0c9c159e2aa04341cdd3e924f44061311060"
MODEL_REPO = "Stanford-ILIAD/prism-qwen25-extra-dinosiglip-224px-0_5b"
INSTRUCTION = "stack the three colored cubes"
SAMPLE_HZ = 30.0         # Native IK frame rate; conversion rejects other rates.
MAX_STEPS = 10_000       # Use 50 first for a quick end-to-end test.
SAVE_FREQ = 1_000
VAL_FRACTION = 0.1       # Hold out complete episodes, never individual frames.
VAL_SPLIT_SEED = 20260920
VAL_FREQ = 250           # Validate every 250 optimizer steps.
VAL_TIME_LIMIT = 60      # Bound validation overhead between training windows.
BATCH_SIZE = 1           # Conservative starting point for full fine-tuning.
GRAD_ACCUM_STEPS = 8     # Effective batch size: 1 x 8 = 8.
LEARNING_RATE = 2e-5     # Full-model learning rate; tune against held-out loss.
SAVE_TO_DRIVE = True     # Checkpoints must outlive the runtime to be resumable.
DRIVE_OUTPUT = "/content/drive/MyDrive/panthera-ik3-30hz-vla-outputs"
SHUTDOWN_WHEN_DONE = False  # True releases the Colab runtime after the export cell.
RESUME_RUN_ID = ""       # A full-finetuning RUN_ID; old LoRA runs cannot be resumed.
RESUME_LEARNING_RATE = None  # For example, 1e-5 overrides the checkpoint LR on resume.
WANDB_ENTITY = ""        # Your W&B user or team. Empty keeps logging offline.
WANDB_PROJECT = "panthera-ik3-30hz"


In [ ]:
# GPU and disk preflight
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU detected. Enable a GPU runtime before continuing.")
subprocess.run(["nvidia-smi"], check=True)

# VLA-Adapter casts weights, inputs and autocast to bfloat16 in a dozen places
# and carries no GradScaler, so a pre-Ampere GPU cannot run it unconverted.
try:
    import torch
    capability = torch.cuda.get_device_capability()
except Exception:
    capability = None
if capability is None:
    print("Could not read compute capability; skipping the bfloat16 check.")
elif capability[0] < 8:
    raise RuntimeError(
        f"{torch.cuda.get_device_name(0)} is compute capability {capability[0]}.{capability[1]}, "
        "which has no bfloat16 support. Choose an L4 or A100 runtime.")
else:
    print(f"Compute capability {capability[0]}.{capability[1]}: bfloat16 supported.")

free_gb = shutil.disk_usage("/content").free / 2**30
print(f"Free runtime disk: {free_gb:.1f} GiB")
if free_gb < 100:
    raise RuntimeError("At least 100 GiB of free runtime disk is recommended.")


In [ ]:
# Mount Drive first: the permission dialog blocks until it is clicked, and
# waiting to ask until the training cell would stall an unattended Run All
# behind an hour of setup. Mounting is idempotent, so the training cell's own
# call is a no-op after this.
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Grant the Drive prompt now; the rest of the notebook runs unattended.")


## 1. Fetch code and create an isolated Python 3.10 environment

VLA-Adapter pins PyTorch 2.2 and TensorFlow 2.15. The separate environment avoids conflicts with Colab's preinstalled packages. Long setup output is normal.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path("/content")
CODE_DIR = ROOT / "RobotArmLearning"
VLA_DIR = CODE_DIR / "VLA-Adapter"
VENV = ROOT / "vla-env"
PYTHON = str(VENV / "bin/python")

def run(command, **kwargs):
    """Run a subprocess, streaming its output into the notebook.

    A child process writes to file descriptor 1, which bypasses the
    `sys.stdout` that ipykernel replaces, so `subprocess.run` output is
    invisible here. Reading the pipe ourselves keeps progress lines -- and
    failures -- in the cell where they can be seen.
    """
    print("+", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               **kwargs)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, command)

if not CODE_DIR.exists():
    run(["git", "clone", "--depth=1", CODE_REPO, CODE_DIR])
if not VLA_DIR.exists():
    run(["git", "clone", VLA_REPO, VLA_DIR])
run(["git", "checkout", VLA_COMMIT], cwd=VLA_DIR)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
run(["uv", "python", "install", "3.10"])
if not VENV.exists():
    run(["uv", "venv", VENV, "--python", "3.10"])
# protobuf has to land on 4.x: TensorFlow 2.15 caps it below 5, and wandb
# 0.26+ dropped the generated bindings for protobuf 3. Nothing here pins it,
# and current tensorflow_metadata resolves to a release whose *_pb2.py modules
# import google.protobuf.runtime_version -- protobuf 5.27 and later only.
# 1.13.1 is the newest that both predates that and permits 4.x; the tfmd
# releases in between cap protobuf below 4.21 and so break wandb instead.
run(["uv", "pip", "install", "--python", PYTHON, "-e", VLA_DIR,
     "tensorflow-metadata==1.13.1", "protobuf==4.25.9",
     "pyarrow==17.0.0", "numpy<2"])
run([PYTHON, "-c", "import torch, tensorflow as tf, pyarrow; "
     "print('torch', torch.__version__, 'tensorflow', tf.__version__, 'pyarrow', pyarrow.__version__)"])


## 2. Register the IK joint-control dataset with VLA-Adapter

Register two cameras, an 8-D padded joint state, and 7-D absolute joint/gripper actions. End-of-episode action chunks repeat the last absolute target. All action dimensions, including gripper opening in metres, are normalized. The patches also enable full-model training, checkpoint resume, gradient checkpointing, and training-only normalization statistics. Cartesian Z-motion loss weighting is disabled because these labels are joint targets.


In [ ]:
def insert_after(path, anchor, addition, marker):
    text = path.read_text()
    if marker not in text:
        if anchor not in text:
            raise RuntimeError(f"Patch anchor not found in {path}")
        path.write_text(text.replace(anchor, anchor + addition, 1))

oxe = VLA_DIR / "prismatic/vla/datasets/rlds/oxe"
insert_after(
    oxe / "configs.py",
    "OXE_DATASET_CONFIGS = {\n",
    '    "panthera_ik_three_block": {\n'
    '        "image_obs_keys": {"primary": "image", "secondary": None, "wrist": "wrist_image"},\n'
    '        "depth_obs_keys": {"primary": None, "secondary": None, "wrist": None},\n'
    '        "state_obs_keys": ["state"],\n'
    '        "state_encoding": StateEncoding.JOINT,\n'
    '        "action_encoding": ActionEncoding.JOINT_POS,\n'
    '        "aux_kwargs": {"absolute_action_mask": [True] * 7,\n'
    '                       "action_normalization_mask": [True] * 7},\n'
    '    },\n',
    '"panthera_ik_three_block"',
)
insert_after(
    oxe / "mixtures.py",
    "OXE_NAMED_MIXTURES: Dict[str, List[Tuple[str, float]]] = {\n",
    '    "panthera_ik_three_block": [("panthera_ik_three_block", 1.0)],\n',
    '"panthera_ik_three_block"',
)
insert_after(
    oxe / "transforms.py",
    "OXE_STANDARDIZATION_TRANSFORMS = {\n",
    '    "panthera_ik_three_block": lambda trajectory: trajectory,\n',
    '"panthera_ik_three_block"',
)

# Upstream rejects single-arm joint encodings. Only this dataset bypasses
# that guard; its seven absolute dimensions are specified in aux_kwargs above.
materialize = oxe / "materialize.py"
materialize_text = materialize.read_text()
old_guard = '    if dataset_kwargs["action_encoding"] not in ['
new_guard = ('    if dataset_name != "panthera_ik_three_block" and '
             'dataset_kwargs["action_encoding"] not in [')
if old_guard in materialize_text:
    materialize.write_text(materialize_text.replace(old_guard, new_guard, 1))

# The Qwen2.5 backbone asks for FlashAttention 2 unconditionally. VLA-Adapter
# leaves flash_attn to a manual post-install step, and it needs an Ampere or
# newer GPU regardless -- a T4 is Turing and cannot run it at all. Fall back to
# PyTorch SDPA unless both the hardware and the package are present.
base_llm = VLA_DIR / "prismatic/models/backbones/llm/base_llm.py"
insert_after(
    base_llm,
    "        return self.tokenizer.pad_token_id\n\n\n",
    "def flash_attention_2_available() -> bool:\n"
    '    """FlashAttention 2 needs an Ampere-or-newer GPU and the flash_attn package."""\n'
    "    import importlib.util\n"
    "    if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8:\n"
    "        return False\n"
    '    return importlib.util.find_spec("flash_attn") is not None\n\n\n',
    "def flash_attention_2_available",
)
attention = base_llm.read_text()
old_attn = "use_flash_attention_2=use_flash_attention_2 if not self.inference_mode else False,"
new_attn = ("use_flash_attention_2=use_flash_attention_2 and flash_attention_2_available()\n"
            "                if not self.inference_mode else False,")
if old_attn in attention:
    base_llm.write_text(attention.replace(old_attn, new_attn, 1))

finetune = VLA_DIR / "vla-scripts/finetune.py"
insert_after(
    finetune,
    "    use_pro_version: bool = True                             # the version number\n",
    "    use_gradient_checkpointing: bool = False\n",
    "use_gradient_checkpointing: bool",
)
checkpoint_anchor = "    # FiLM setup\n"
checkpoint_block = (
    "    if cfg.use_gradient_checkpointing:\n"
    "        llm = (vla.base_model.model if cfg.use_lora else vla).language_model\n"
    "        llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\"use_reentrant\": False})\n"
    "        llm.config.use_cache = False\n"
    "        print(\"Gradient checkpointing enabled on language_model\")\n\n"
)
text = finetune.read_text()
if "Gradient checkpointing enabled on language_model" not in text:
    if checkpoint_anchor not in text:
        raise RuntimeError("Gradient-checkpointing patch anchor not found")
    text = text.replace(checkpoint_anchor, checkpoint_block + checkpoint_anchor, 1)
old_save = "if gradient_step_idx > 0 and log_step % cfg.save_freq == 0:"
new_save = ("if gradient_step_idx > 0 and log_step % cfg.save_freq == 0 "
            "and (batch_idx + 1) % cfg.grad_accumulation_steps == 0:")
if old_save in text:
    text = text.replace(old_save, new_save, 1)
finetune.write_text(text)

# Compute normalization statistics from training episodes only.
rlds_dataset = VLA_DIR / "prismatic/vla/datasets/rlds/dataset.py"
statistics_text = rlds_dataset.read_text()
statistics_old = (
    '        full_dataset = dl.DLataset.from_rlds(\n'
    '            builder, split="all", shuffle=False, num_parallel_reads=num_parallel_reads\n'
    '        ).traj_map(restructure, num_parallel_calls)\n'
)
statistics_new = (
    '        statistics_split = ("train" if name == "panthera_ik_three_block" '
    'and "val" in builder.info.splits else "all")\n'
    '        full_dataset = dl.DLataset.from_rlds(\n'
    '            builder, split=statistics_split, shuffle=False, num_parallel_reads=num_parallel_reads\n'
    '        ).traj_map(restructure, num_parallel_calls)\n'
)
if "statistics_split =" not in statistics_text:
    if statistics_old not in statistics_text:
        raise RuntimeError("Training-only statistics patch anchor not found")
    statistics_text = statistics_text.replace(statistics_old, statistics_new, 1)
    # Include the selected split in the statistics cache key so an older
    # all-data cache cannot leak validation distribution information.
    statistics_text = statistics_text.replace(
        '                str(builder.info),\n',
        '                str(builder.info),\n                statistics_split,\n',
        1,
    )
    rlds_dataset.write_text(statistics_text)
# Also handle a checkout patched by an earlier teleop notebook run.
statistics_text = rlds_dataset.read_text()
statistics_text = statistics_text.replace(
    'statistics_split = ("train" if name == "robot_arm_learning_panthera" ',
    'statistics_split = ("train" if name in '
    '("robot_arm_learning_panthera", "panthera_ik_three_block") ',
)
rlds_dataset.write_text(statistics_text)
insert_after(
    finetune,
    "    use_gradient_checkpointing: bool = False\n",
    "    balance_z_loss: bool = False\n"
    "    z_neutral_threshold: float = 0.045\n"
    "    z_down_weight: float = 1.0\n"
    "    z_neutral_weight: float = 0.7\n"
    "    z_up_weight: float = 1.8\n",
    "balance_z_loss: bool",
)
# The trainer hardcodes `mode="offline"` and never passes --wandb_entity to
# wandb.init, so neither the flag nor WANDB_MODE can reach it. Honour both.
text = finetune.read_text()
old_init = 'wandb.init(project=cfg.wandb_project, name=f"ft+{run_id}", mode="offline")'
new_init = ('wandb.init(entity=cfg.wandb_entity or None, project=cfg.wandb_project,\n'
            '                   name=f"ft+{run_id}", mode=os.environ.get("WANDB_MODE", "offline"))')
if old_init in text:
    finetune.write_text(text.replace(old_init, new_init, 1))

# Make the upstream validation path deterministic and compatible with the
# project-specific forward-pass configuration.
text = finetune.read_text()
old_val_dataset = '            image_aug=cfg.image_aug,\n            train=False,\n'
new_val_dataset = '            image_aug=False,\n            train=False,\n'
if old_val_dataset in text:
    text = text.replace(old_val_dataset, new_val_dataset, 1)
old_val_call = (
    '                use_pro_version=cfg.use_pro_version\n'
    '            )\n\n            # Add the loss value to the metrics\n'
)
new_val_call = (
    '                use_pro_version=cfg.use_pro_version,\n'
    '                cfg=cfg,\n'
    '            )\n\n            # Add the loss value to the metrics\n'
)
if old_val_call in text:
    text = text.replace(old_val_call, new_val_call, 1)
old_val_condition = (
    '            if cfg.use_val_set and log_step > 0 and log_step % cfg.val_freq == 0:\n'
)
new_val_condition = (
    '            if (cfg.use_val_set and log_step > 0 and log_step % cfg.val_freq == 0\n'
    '                    and (batch_idx + 1) % cfg.grad_accumulation_steps == 0):\n'
)
if old_val_condition in text:
    text = text.replace(old_val_condition, new_val_condition, 1)
old_val_mode = '    val_start_time = time.time()\n    vla.eval()\n'
new_val_mode = (
    '    val_start_time = time.time()\n'
    '    previous_phase = cfg.phase\n'
    '    cfg.phase = "Inference"\n'
    '    vla.eval()\n'
    '    if action_head is not None:\n'
    '        action_head.eval()\n'
    '    if proprio_projector is not None:\n'
    '        proprio_projector.eval()\n'
)
if old_val_mode in text:
    text = text.replace(old_val_mode, new_val_mode, 1)
old_val_end = (
    '    if distributed_state.is_main_process:\n'
    '        log_metrics_to_wandb(avg_val_metrics, "VLA Val", log_step, wandb)\n\n\n'
)
new_val_end = (
    '    if distributed_state.is_main_process:\n'
    '        log_metrics_to_wandb(avg_val_metrics, "VLA Val", log_step, wandb)\n\n'
    '    cfg.phase = previous_phase\n'
    '    if action_head is not None:\n'
    '        action_head.train()\n'
    '    if proprio_projector is not None:\n'
    '        proprio_projector.train()\n\n\n'
)
if old_val_end in text:
    text = text.replace(old_val_end, new_val_end, 1)
finetune.write_text(text)

# --- Resume support -------------------------------------------------------
# Upstream's --resume saves no optimizer or scheduler state and
# reads component checkpoints under a step-numbered name that
# --save_latest_checkpoint_only never writes. These patches add a
# self-contained --resume_checkpoint <run_dir> and leave --resume alone.
def replace_once(path, old, new, marker):
    text = path.read_text()
    if marker in text:
        return
    if old not in text:
        raise RuntimeError(f"Patch anchor not found in {path}")
    path.write_text(text.replace(old, new, 1))

replace_once(
    finetune,
    "        loss = torch.nn.L1Loss()(predicted_actions, ground_truth_actions)\n",
    "        if cfg.balance_z_loss:\n"
    "            per_timestep_l1 = torch.abs(\n"
    "                predicted_actions.float() - ground_truth_actions.float()\n"
    "            ).mean(dim=-1)\n"
    "            dz = ground_truth_actions[..., 2].float()\n"
    "            weights = torch.where(\n"
    "                dz > cfg.z_neutral_threshold, cfg.z_up_weight,\n"
    "                torch.where(dz < -cfg.z_neutral_threshold,\n"
    "                            cfg.z_down_weight, cfg.z_neutral_weight),\n"
    "            )\n"
    "            weights = weights / weights.mean().detach().clamp_min(1e-6)\n"
    "            loss = (per_timestep_l1 * weights).mean()\n"
    "        else:\n"
    "            loss = torch.nn.L1Loss()(predicted_actions, ground_truth_actions)\n",
    "per_timestep_l1 = torch.abs",
)

insert_after(
    finetune,
    "    z_up_weight: float = 1.8\n",
    '    resume_checkpoint: str = ""\n',
    "resume_checkpoint: str",
)
insert_after(
    finetune,
    '    resume_checkpoint: str = ""\n',
    "    resume_learning_rate: float = -1.0\n",
    "resume_learning_rate: float",
)

# The trained LoRA weights, which upstream's resume path drops on the floor.
replace_once(
    finetune,
    "        vla = get_peft_model(vla, lora_config)\n",
    "        if cfg.resume_checkpoint:\n"
    '            resume_adapter_dir = os.path.join(cfg.resume_checkpoint, "lora_adapter")\n'
    '            print(f"Resuming LoRA adapter from {resume_adapter_dir}")\n'
    "            vla = PeftModel.from_pretrained(vla, resume_adapter_dir, is_trainable=True)\n"
    "        else:\n"
    "            vla = get_peft_model(vla, lora_config)\n",
    "Resuming LoRA adapter from",
)

# Action head, proprio projector and (with FiLM) vision backbone. Component
# files are named either "<module>--latest_checkpoint.pt" or
# "<module>--<step>_checkpoint.pt" depending on save_latest_checkpoint_only.
replace_once(
    finetune,
    "    if cfg.resume:\n"
    "        state_dict = load_checkpoint(module_name, cfg.resum_vla_path, cfg.resume_step)\n"
    "        module.load_state_dict(state_dict)\n"
    "        print('loaded!!!!!!!!!')\n",
    "    if cfg.resume_checkpoint:\n"
    "        directory = Path(cfg.resume_checkpoint)\n"
    '        latest = directory / f"{module_name}--latest_checkpoint.pt"\n'
    "        if not latest.exists():\n"
    '            numbered = sorted(directory.glob(f"{module_name}--*_checkpoint.pt"),\n'
    "                              key=os.path.getmtime)\n"
    "            if not numbered:\n"
    '                raise FileNotFoundError(f"No {module_name} checkpoint in {directory}")\n'
    "            latest = numbered[-1]\n"
    '        print(f"Resuming {module_name} from {latest}")\n'
    "        module.load_state_dict(remove_ddp_in_checkpoint(\n"
    '            torch.load(latest, weights_only=True, map_location="cpu")))\n'
    "    elif cfg.resume:\n"
    "        state_dict = load_checkpoint(module_name, cfg.resum_vla_path, cfg.resume_step)\n"
    "        module.load_state_dict(state_dict)\n"
    "        print('loaded!!!!!!!!!')\n",
    "Resuming {module_name} from",
)

# `action_queries` is trained but lives outside the LoRA adapter, so
# PeftModel.save_pretrained drops it -- upstream only rescues it when merging.
insert_after(
    finetune,
    "    # Wrap VLA with DDP\n",
    "    if cfg.resume_checkpoint and cfg.use_lora:\n"
    '        extras_path = Path(cfg.resume_checkpoint) / "trainable_extras--latest_checkpoint.pt"\n'
    "        if extras_path.exists():\n"
    "            extras = remove_ddp_in_checkpoint(\n"
    '                torch.load(extras_path, weights_only=True, map_location="cpu"))\n'
    "            unexpected = vla.load_state_dict(extras, strict=False).unexpected_keys\n"
    "            if unexpected:\n"
    '                raise RuntimeError(f"Unexpected keys in {extras_path}: {unexpected}")\n'
    '            print(f"Resumed {len(extras)} trainable tensors outside the LoRA adapter")\n'
    "        else:\n"
    '            print(f"No {extras_path.name} to resume; action_queries start from the base model")\n',
    "trainable_extras--latest_checkpoint.pt",
)

# Optimizer moments and the LR schedule position. Without these a resumed run
# restarts AdamW cold and counts the num_steps_before_decay milestone from the
# resume point rather than from the start of training.
insert_after(
    finetune,
    "    scheduler = MultiStepLR(\n"
    "        optimizer,\n"
    "        milestones=[cfg.num_steps_before_decay],  # Number of steps after which LR will change\n"
    "        gamma=0.1,  # Multiplicative factor of learning rate decay\n"
    "    )\n",
    "\n"
    "    resume_step = 0\n"
    "    if cfg.resume_checkpoint:\n"
    '        training_state_path = Path(cfg.resume_checkpoint) / "training_state--latest_checkpoint.pt"\n'
    "        if not training_state_path.exists():\n"
    "            raise FileNotFoundError(\n"
    '                f"{training_state_path} is missing; that checkpoint predates resume support."\n'
    "            )\n"
    "        # weights_only=False: MultiStepLR's state holds a collections.Counter.\n"
    '        training_state = torch.load(training_state_path, map_location="cpu", weights_only=False)\n'
    '        expected_mode = "lora" if cfg.use_lora else "full"\n'
    '        if training_state.get("finetuning_mode", "lora") != expected_mode:\n'
    '            raise RuntimeError("Cannot resume across LoRA/full fine-tuning modes; start a fresh run.")\n'
    '        optimizer.load_state_dict(training_state["optimizer"])\n'
    '        scheduler.load_state_dict(training_state["scheduler"])\n'
    "        if cfg.resume_learning_rate > 0:\n"
    "            for param_group in optimizer.param_groups:\n"
    "                param_group[\"lr\"] = cfg.resume_learning_rate\n"
    "                param_group[\"initial_lr\"] = cfg.resume_learning_rate\n"
    "            scheduler.base_lrs = [cfg.resume_learning_rate] * len(optimizer.param_groups)\n"
    "            scheduler._last_lr = [cfg.resume_learning_rate] * len(optimizer.param_groups)\n"
    '            print(f"Overrode resumed learning rate to {cfg.resume_learning_rate:g}")\n'
    '        resume_step = training_state["step"]\n'
    '        print(f"Resuming optimizer and LR schedule at step {resume_step}")\n',
    "resume_step = 0",
)
# Add LR override support to runtimes patched by an older notebook version.
insert_after(
    finetune,
    '        scheduler.load_state_dict(training_state["scheduler"])\n',
    "        if cfg.resume_learning_rate > 0:\n"
    "            for param_group in optimizer.param_groups:\n"
    "                param_group[\"lr\"] = cfg.resume_learning_rate\n"
    "                param_group[\"initial_lr\"] = cfg.resume_learning_rate\n"
    "            scheduler.base_lrs = [cfg.resume_learning_rate] * len(optimizer.param_groups)\n"
    "            scheduler._last_lr = [cfg.resume_learning_rate] * len(optimizer.param_groups)\n"
    '            print(f"Overrode resumed learning rate to {cfg.resume_learning_rate:g}")\n',
    "Overrode resumed learning rate to",
)

# Steps stay absolute across a resume, so max_steps, save_freq and the W&B
# x-axis all mean the same thing they did in the original run.
replace_once(
    finetune,
    "            log_step = gradient_step_idx if not cfg.resume else cfg.resume_step + gradient_step_idx\n",
    "            log_step = resume_step + (\n"
    "                gradient_step_idx if not cfg.resume else cfg.resume_step + gradient_step_idx)\n",
    "log_step = resume_step + (",
)
replace_once(
    finetune,
    "with tqdm.tqdm(total=cfg.max_steps, leave=False) as progress:",
    "with tqdm.tqdm(total=cfg.max_steps, initial=resume_step, leave=False) as progress:",
    "initial=resume_step",
)

# Write the training state next to every checkpoint the trainer saves.
insert_after(
    finetune,
    "                    new_state_dict=RAW_STATE_DICT,\n"
    "                )\n",
    "                if distributed_state.is_main_process:\n"
    "                    state_dir = Path(run_dir) if cfg.save_latest_checkpoint_only \\\n"
    '                        else Path(str(run_dir) + f"--{log_step}_chkpt")\n'
    "                    torch.save(\n"
    '                        {"optimizer": optimizer.state_dict(),\n'
    '                         "scheduler": scheduler.state_dict(),\n'
    '                         "step": log_step,\n'
    '                         "finetuning_mode": "lora" if cfg.use_lora else "full"},\n'
    '                        state_dir / "training_state--latest_checkpoint.pt",\n'
    "                    )\n"
    "                    if cfg.use_lora:\n"
    "                        torch.save(\n"
    '                            {k: v for k, v in vla.state_dict().items() if "action_queries" in k},\n'
    '                            state_dir / "trainable_extras--latest_checkpoint.pt",\n'
    "                        )\n",
    '"training_state--latest_checkpoint.pt",',
)

# draccus turns an empty string on the command line into the literal "None",
# which is truthy: a fresh run would take the resume path and look for the
# adapter on the Hugging Face Hub. Normalise the optional flags before
# anything reads them.
insert_after(
    finetune,
    '    cfg.config_file_path = cfg.config_file_path.rstrip("/")\n',
    '    for _optional in ("resume_checkpoint", "wandb_entity"):\n'
    '        if getattr(cfg, _optional) in (None, "", "None", "none"):\n'
    '            setattr(cfg, _optional, "")\n',
    'for _optional in ("resume_checkpoint", "wandb_entity")',
)

# Full fine-tuning: unfreeze every backbone parameter and save/load a complete
# Hugging Face model, including action_queries. No PEFT wrapper is created.
replace_once(
    finetune,
    "    else:\n"
    "        for name, param in vla.named_parameters():\n"
    '            if "action_queries" in name:\n'
    "                param.requires_grad = True\n",
    "    else:\n"
    "        vla.requires_grad_(True)\n"
    "        total = sum(p.numel() for p in vla.parameters())\n"
    "        trainable = sum(p.numel() for p in vla.parameters() if p.requires_grad)\n"
    '        print(f"Full fine-tuning: {trainable:,}/{total:,} VLA parameters trainable")\n'
    "        assert trainable == total\n",
    "Full fine-tuning:",
)
replace_once(
    finetune,
    "    if cfg.use_minivlm:\n        hf_token = ''\n",
    "    if cfg.resume_checkpoint and not cfg.use_lora:\n"
    "        RAW_STATE_DICT = {}\n"
    "        vla = AutoModelForVision2Seq.from_pretrained(\n"
    "            cfg.resume_checkpoint, torch_dtype=torch.bfloat16,\n"
    "            low_cpu_mem_usage=False, trust_remote_code=False,\n"
    "        ).to(device_id)\n"
    '        print(f"Resumed full VLA from {cfg.resume_checkpoint}")\n'
    "    elif cfg.use_minivlm:\n        hf_token = ''\n",
    "Resumed full VLA from",
)
replace_once(
    finetune,
    "        if cfg.use_fz:\n"
    "            vla.module.save_pretrained(checkpoint_dir) # directly save checkpoint without lora\n",
    "        if not cfg.use_lora:\n"
    "            vla.module.save_pretrained(checkpoint_dir)  # Complete trainable VLA\n",
    "# Complete trainable VLA",
)
replace_once(
    finetune,
    "        os.makedirs(adapter_dir, exist_ok=True)\n",
    "        if cfg.use_lora:\n            os.makedirs(adapter_dir, exist_ok=True)\n",
    "if cfg.use_lora:\n            os.makedirs(adapter_dir",
)
# The source VLM and remapped weights are needed only for initialization when
# full fine-tuning; retaining them wastes host memory for the entire run.
replace_once(
    finetune,
    "        del old_state_dict\n",
    "        del old_state_dict\n"
    "        if not cfg.use_lora:\n"
    "            del vlm\n"
    "            RAW_STATE_DICT = {}  # No LoRA merge needs the original weights\n",
    "# No LoRA merge needs the original weights",
)

# Upgrade resume patches already installed by an earlier LoRA notebook.
insert_after(
    finetune,
    '        training_state = torch.load(training_state_path, map_location="cpu", weights_only=False)\n',
    '        expected_mode = "lora" if cfg.use_lora else "full"\n'
    '        if training_state.get("finetuning_mode", "lora") != expected_mode:\n'
    '            raise RuntimeError("Cannot resume across LoRA/full fine-tuning modes; start a fresh run.")\n',
    'expected_mode = "lora"',
)
text = finetune.read_text()
text = text.replace(
    '    if cfg.resume_checkpoint:\n        extras_path =',
    '    if cfg.resume_checkpoint and cfg.use_lora:\n        extras_path =',
    1,
)
text = text.replace(
    '                         "step": log_step},\n',
    '                         "step": log_step,\n'
    '                         "finetuning_mode": "lora" if cfg.use_lora else "full"},\n',
    1,
)
old_extras_save = (
    '                    torch.save(\n'
    '                        {k: v for k, v in vla.state_dict().items() if "action_queries" in k},\n'
    '                        state_dir / "trainable_extras--latest_checkpoint.pt",\n'
    '                    )\n'
)
text = text.replace(
    old_extras_save,
    '                    if cfg.use_lora:\n' + ''.join(
        '    ' + line for line in old_extras_save.splitlines(keepends=True)),
    1,
)
finetune.write_text(text)

print("RobotArmLearning full fine-tuning setup is ready.")


## 3. Download and verify the published IK dataset

Fetch the pinned LeRobot Parquet export and metadata, including the embedded shoulder/wrist JPEGs. Verify publication completeness, frame rate, episode/frame counts, and file hashes. Downloads resume from the Hugging Face cache.


In [ ]:
IK_DATA_DIR = ROOT / "datasets" / ("panthera-ik3-30hz-" + DATASET_REVISION)
download_program = f'''
from huggingface_hub import snapshot_download
from pathlib import Path
import hashlib
import json
snapshot_download(
    repo_id={DATASET_REPO!r}, repo_type="dataset", revision={DATASET_REVISION!r},
    local_dir={str(IK_DATA_DIR)!r},
    allow_patterns=["data/**", "meta/**", "COMPLETE", "verification.json", "upload_manifest.json"],
)
root = Path({str(IK_DATA_DIR)!r})
if not (root / "COMPLETE").is_file():
    raise RuntimeError("IK dataset publication is incomplete")
info = json.loads((root / "meta/info.json").read_text())
if (info["fps"], info["total_episodes"], info["total_frames"]) != (30, 1000, 862766):
    raise RuntimeError("Expected the 1,000-demo, 862,766-frame 30 Hz IK dataset")
if {SAMPLE_HZ!r} != info["fps"]:
    raise RuntimeError("SAMPLE_HZ must equal the native IK dataset rate of 30 Hz")
manifest = json.loads((root / "upload_manifest.json").read_text())
for name, entry in manifest["files"].items():
    if not name.startswith(("data/", "meta/")):
        continue
    path = root / name
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(block)
    if path.stat().st_size != entry["size"] or digest.hexdigest() != entry["sha256"]:
        raise RuntimeError(f"Dataset file failed verification: {{name}}")
print("Verified 1,000 IK episodes / 862,766 frames at 30 Hz")
'''
run([PYTHON, "-c", download_program])


## 4. Convert the existing camera images and joint targets to RLDS

The converter streams Parquet shards and buffers one episode at a time. It keeps the original JPEG bytes, every 30 Hz frame, and the already-shifted actions. A deterministic 10% episode holdout is used by default. Source revision and split settings have their own cache directory. The converter is embedded below so this notebook can run without a separately updated project checkout.


In [ ]:
# Embedded copy of teleop/build_panthera_ik_rlds.py.
IK_BUILDER = ROOT / "build_panthera_ik_rlds.py"
IK_BUILDER_SOURCE = r'''"""Convert the published 30 Hz IK LeRobot Parquet dataset to VLA-Adapter RLDS.

Keep embedded JPEGs and already-shifted absolute joint/gripper actions unchanged.
Only proprio gains one zero padding element before the gripper for VLA's 8-D input.
"""
from __future__ import annotations

import argparse
import itertools
import json
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import tensorflow_datasets as tfds

CAMERAS = ("observation.images.shoulder", "observation.images.wrist")


def dataset_info(root, hz=30):
    root = Path(root)
    if not (root / "COMPLETE").is_file():
        raise ValueError("IK dataset publication is incomplete")
    info = json.loads((root / "meta/info.json").read_text())
    if info["fps"] != hz or hz != 30:
        raise ValueError("The IK dataset must stay at its native 30 Hz; no resampling")
    if info["codebase_version"] != "v3.0" or info["total_episodes"] < 2:
        raise ValueError("Expected a LeRobot v3 dataset with at least two episodes")
    for key in ("action", "observation.state"):
        if info["features"][key]["shape"] != [7]:
            raise ValueError(f"Expected six joints and gripper for {key}")
    for key in CAMERAS:
        if info["features"][key]["shape"] != [256, 256, 3]:
            raise ValueError(f"Expected 256x256 RGB for {key}")
    return info


def iter_episodes(root, info):
    """Stream across shard/batch boundaries, buffering at most one episode."""
    columns = ["episode_index", "frame_index", "index", "timestamp",
               "observation.state", "action", *CAMERAS]
    paths = sorted((Path(root) / "data").rglob("*.parquet"))
    if not paths:
        raise ValueError("No IK Parquet shards found")

    def rows():
        for path in paths:
            for batch in pq.ParquetFile(path).iter_batches(batch_size=64, columns=columns):
                yield from batch.to_pylist()

    total = 0
    episodes = 0
    for episode_id, group in itertools.groupby(rows(), key=lambda row: row["episode_index"]):
        episode = list(group)
        if episode_id != episodes:
            raise ValueError("Expected contiguous ordered episode indices")
        for frame, row in enumerate(episode):
            if row["frame_index"] != frame or row["index"] != total + frame:
                raise ValueError(f"Missing or reordered frames in episode {episode_id}")
            if not np.isclose(row["timestamp"], frame / 30, atol=1e-5, rtol=0):
                raise ValueError(f"Frame timing is not 30 Hz in episode {episode_id}")
        total += len(episode)
        episodes += 1
        yield episode_id, episode
    if total != info["total_frames"] or episodes != info["total_episodes"]:
        raise ValueError("IK frame/episode counts do not match dataset metadata")


def episode_steps(rows, instruction):
    states = np.asarray([row["observation.state"] for row in rows], dtype=np.float32)
    actions = np.asarray([row["action"] for row in rows], dtype=np.float32)
    if (states.shape != (len(rows), 7) or actions.shape != (len(rows), 7)
            or not np.isfinite(states).all() or not np.isfinite(actions).all()):
        raise ValueError("Invalid joint state/action vectors")
    # Six measured joints, one padding joint, current opening command in metres.
    proprio = np.insert(states, 6, 0, axis=1)
    steps = []
    for index, row in enumerate(rows):
        images = [row[key]["bytes"] for key in CAMERAS]
        if not all(isinstance(value, bytes) and value for value in images):
            raise ValueError("Expected embedded JPEG bytes for both cameras")
        last = index == len(rows) - 1
        steps.append({
            "observation": {"image": images[0], "wrist_image": images[1],
                            "state": proprio[index]},
            # Labels already point to t+1/30 s. Never shift or delta-convert again.
            "action": actions[index],
            "language_instruction": instruction,
            "is_first": index == 0, "is_last": last, "is_terminal": last,
            "reward": np.float32(last), "discount": np.float32(not last),
        })
    return steps


class PantheraIkThreeBlock(tfds.core.GeneratorBasedBuilder):
    """Scripted three-block IK demonstrations, absolute joint control at 30 Hz."""

    VERSION = tfds.core.Version("1.0.0")
    RELEASE_NOTES = {"1.0.0": "Native 30 Hz joint actions and episode-level holdout."}
    # TFDS 4.9.3 cannot infer resource paths for the teleop namespace package.
    code_path = Path(__file__)

    def __init__(self, *args, dataset_dir, instruction="stack the three colored cubes",
                 hz=30, val_fraction=0.1, split_seed=20260920, **kwargs):
        self.dataset_dir = Path(dataset_dir)
        self.source_info = dataset_info(self.dataset_dir, hz)
        self.instruction = instruction
        if not 0 < val_fraction < 1:
            raise ValueError("val_fraction must be between 0 and 1")
        count = self.source_info["total_episodes"]
        val_count = min(count - 1, max(1, round(count * val_fraction)))
        shuffled = np.random.default_rng(split_seed).permutation(count)
        self.val_episodes = set(shuffled[:val_count].tolist())
        super().__init__(*args, **kwargs)

    def _info(self):
        return tfds.core.DatasetInfo(builder=self, features=tfds.features.FeaturesDict({
            "steps": tfds.features.Dataset({
                "observation": {
                    "image": tfds.features.Image(shape=(256, 256, 3), encoding_format="jpeg"),
                    "wrist_image": tfds.features.Image(shape=(256, 256, 3), encoding_format="jpeg"),
                    "state": tfds.features.Tensor(shape=(8,), dtype=np.float32),
                },
                "action": tfds.features.Tensor(shape=(7,), dtype=np.float32),
                "language_instruction": tfds.features.Text(),
                "is_first": np.bool_, "is_last": np.bool_, "is_terminal": np.bool_,
                "reward": np.float32, "discount": np.float32,
            }),
            "episode_metadata": {"episode_index": np.int64},
        }), description=__doc__)

    def _split_generators(self, dl_manager):
        del dl_manager
        return {"train": self._generate_examples(False), "val": self._generate_examples(True)}

    def _generate_examples(self, validation):
        for episode_id, rows in iter_episodes(self.dataset_dir, self.source_info):
            if (episode_id in self.val_episodes) != validation:
                continue
            yield str(episode_id), {
                "steps": episode_steps(rows, self.instruction),
                "episode_metadata": {"episode_index": episode_id},
            }


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset-dir", type=Path, required=True)
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--hz", type=float, default=30)
    parser.add_argument("--instruction", default="stack the three colored cubes")
    parser.add_argument("--val-fraction", type=float, default=0.1)
    parser.add_argument("--split-seed", type=int, default=20260920)
    args = parser.parse_args()
    builder = PantheraIkThreeBlock(
        data_dir=str(args.data_dir), dataset_dir=args.dataset_dir, hz=args.hz,
        instruction=args.instruction, val_fraction=args.val_fraction, split_seed=args.split_seed)
    # Refuse stale TFRecords with a different split/source, even for direct CLI use.
    contract = {"source": str(args.dataset_dir.resolve()), "hz": args.hz,
                "instruction": args.instruction, "val_fraction": args.val_fraction,
                "split_seed": args.split_seed, "version": str(builder.VERSION)}
    args.data_dir.mkdir(parents=True, exist_ok=True)
    contract_path = args.data_dir / "conversion.json"
    if contract_path.exists() and json.loads(contract_path.read_text()) != contract:
        raise ValueError("RLDS conversion settings changed; choose a fresh --data-dir")
    contract_path.write_text(json.dumps(contract, indent=2) + "\n")
    builder.download_and_prepare()
    print(f"Built {builder.info.full_name} at {builder.data_dir}")
    print(builder.info.splits)


if __name__ == "__main__":
    main()
'''
IK_BUILDER.write_text(IK_BUILDER_SOURCE)


In [ ]:
RLDS_DIR = VLA_DIR / "data" / (
    f"ik3-30hz-{DATASET_REVISION}-val{VAL_FRACTION}-seed{VAL_SPLIT_SEED}")
run([PYTHON, IK_BUILDER,
     "--dataset-dir", IK_DATA_DIR, "--data-dir", RLDS_DIR,
     "--hz", str(SAMPLE_HZ), "--instruction", INSTRUCTION,
     "--val-fraction", str(VAL_FRACTION), "--split-seed", str(VAL_SPLIT_SEED)])


In [ ]:
# Preview the exact stored camera pair without installing PyArrow in Colab's kernel.
from IPython.display import display, Image
preview_program = f'''
from pathlib import Path
import pyarrow.parquet as pq
root = Path({str(IK_DATA_DIR)!r})
path = sorted((root / "data").rglob("*.parquet"))[0]
columns = ["observation.images.shoulder", "observation.images.wrist"]
row = next(pq.ParquetFile(path).iter_batches(batch_size=1, columns=columns)).to_pylist()[0]
for camera, key in zip(("shoulder", "wrist"), columns):
    (root / (camera + "_preview.jpg")).write_bytes(row[key]["bytes"])
'''
run([PYTHON, "-c", preview_program])
for camera in ("shoulder", "wrist"):
    display(Image(filename=str(IK_DATA_DIR / (camera + "_preview.jpg")), width=320))


## 5. Download the base model

In [ ]:
MODEL_DIR = VLA_DIR / "pretrained_models/prism-qwen25-extra-dinosiglip-224px-0_5b"
model_program = f'''
from huggingface_hub import snapshot_download
snapshot_download(repo_id={MODEL_REPO!r}, local_dir={str(MODEL_DIR)!r})
'''
run([PYTHON, "-c", model_program])


## 6. Fine-tune

Full fine-tuning updates every VLA parameter, including both vision backbones, the language model, multimodal projector, and action queries, plus the action head and proprio projector. LoRA is disabled. The defaults use an effective batch size of 8 (`batch_size=1`, eight accumulation steps), language-model gradient checkpointing, and `LEARNING_RATE=2e-5`. Start with an A100; memory use has not been benchmarked for this configuration. Every `VAL_FREQ` optimizer steps, evaluation runs without image augmentation on the held-out episodes and logs `VLA Val/Loss`, `VLA Val/Current Action L1 Loss`, and `VLA Val/Next Actions L1 Loss`. The last metric is the direct counterpart to the training curve. `VAL_TIME_LIMIT` bounds the added runtime. If CUDA reports an out-of-memory error at batch size 1, use a GPU with more memory; increasing accumulation alone does not reduce memory at that point. Set `WANDB_ENTITY` in the settings cell to log to wandb.ai; leaving it empty keeps runs offline in `VLA-Adapter/wandb/`. An online run needs an API key from <https://wandb.ai/authorize>, taken from the `WANDB_API_KEY` Colab secret (key icon in the sidebar, with notebook access enabled) or prompted for if that secret is missing. On an Ampere-or-newer GPU, `uv pip install --python PYTHON flash_attn==2.5.5` is picked up automatically by the attention patch and speeds up training; it is optional. With `SAVE_TO_DRIVE=False`, run the export cell before the Colab runtime disconnects.

This cell prints its `RUN_ID`. A checkpoint trained before this validation split existed has already seen every episode, so it cannot produce an honest held-out loss; leave `RESUME_RUN_ID` empty and start one fresh run. After that, interrupted split-aware runs can be continued by setting `RESUME_RUN_ID` to their value and rerunning from the top: training restores the complete VLA model (including action queries), action head, proprio projector, AdamW moments and LR schedule position from the last checkpoint, and step numbers stay absolute, so `MAX_STEPS` still means total steps. The notebook records and checks the split configuration before allowing a resume. Only full-finetuning checkpoints from this notebook can be resumed; old LoRA runs require a fresh `RUN_ID`. Resume requires the full model weights, `config.json`, component checkpoints, and `training_state--latest_checkpoint.pt`. The RLDS input pipeline restarts at the head of its shuffle stream rather than mid-epoch.
These checkpoints predict absolute joint targets at 30 Hz. The legacy `rollout.py` expects end-effector deltas and is not compatible with this action contract.


In [ ]:
from datetime import datetime, timezone
import json

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")  # Already mounted above; returns immediately.
    OUTPUT_ROOT = Path(DRIVE_OUTPUT)
else:
    OUTPUT_ROOT = VLA_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = RESUME_RUN_ID or (
    "panthera-ik3-30hz-full-colab-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
RESUME_DIR = OUTPUT_ROOT / RUN_ID if RESUME_RUN_ID else None
def require_full_checkpoint(directory):
    required = ["config.json", "training_state--latest_checkpoint.pt",
                "action_head--latest_checkpoint.pt", "proprio_projector--latest_checkpoint.pt"]
    missing = [name for name in required if not (directory / name).is_file()]
    has_weights = any((directory / name).is_file() for name in (
        "model.safetensors", "model.safetensors.index.json",
        "pytorch_model.bin", "pytorch_model.bin.index.json"))
    if missing or not has_weights:
        raise RuntimeError(
            f"No complete full-finetuning checkpoint at {directory}. Missing: {missing}; "
            f"full model weights present: {has_weights}. Old LoRA runs cannot be resumed; "
            "leave RESUME_RUN_ID empty to start fresh.")

if RESUME_DIR is not None:
    require_full_checkpoint(RESUME_DIR)
validation_config = {
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "rlds_version": "1.0.0",
    "dataset_name": "panthera_ik_three_block",
    "sample_hz": SAMPLE_HZ,
    "action_contract": "absolute_next_joint_targets_radians_gripper_metres",
    "instruction": INSTRUCTION,
    "val_fraction": VAL_FRACTION,
    "split_seed": VAL_SPLIT_SEED,
}
validation_config_path = OUTPUT_ROOT / RUN_ID / "validation_split.json"
if RESUME_DIR is not None:
    if not validation_config_path.exists():
        raise RuntimeError(
            "This run predates the held-out validation split and has seen all episodes. "
            "Leave RESUME_RUN_ID empty and start a fresh run for valid metrics.")
    saved_validation_config = json.loads(validation_config_path.read_text())
    if saved_validation_config != validation_config:
        raise RuntimeError(
            f"Dataset/action contract or validation split changed since this run began; start a fresh run: "
            f"{saved_validation_config} != {validation_config}")
else:
    validation_config_path.parent.mkdir(parents=True, exist_ok=True)
    validation_config_path.write_text(json.dumps(validation_config, indent=2) + "\n")
print("RUN_ID:", RUN_ID, "(resuming)" if RESUME_DIR else "(new run)")

WANDB_MODE = "online" if WANDB_ENTITY else "offline"
if WANDB_ENTITY and not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        import getpass
        os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key (wandb.ai/authorize): ")

command = [
    VENV / "bin/torchrun", "--standalone", "--nnodes", "1", "--nproc-per-node", "1",
    VLA_DIR / "vla-scripts/finetune.py",
    "--vlm_path", MODEL_DIR,
    "--config_file_path", VLA_DIR / "pretrained_models/configs",
    "--data_root_dir", RLDS_DIR,
    "--dataset_name", "panthera_ik_three_block",
    "--run_root_dir", OUTPUT_ROOT,
    "--run_id_override", RUN_ID,
    "--use_film", "False",
    "--num_images_in_input", "2",
    "--use_proprio", "True",
    "--use_lora", "False",
    "--use_fz", "True",
    "--use_minivlm", "True",
    "--image_aug", "True",
    "--shuffle_buffer_size", "12000",
    "--use_val_set", "True",
    "--val_freq", str(VAL_FREQ),
    "--val_time_limit", str(VAL_TIME_LIMIT),
    "--num_steps_before_decay", str(max(1, int(MAX_STEPS * 0.8))),
    "--max_steps", str(MAX_STEPS),
    "--save_freq", str(SAVE_FREQ),
    "--save_latest_checkpoint_only", "True",
    "--batch_size", str(BATCH_SIZE),
    "--grad_accumulation_steps", str(GRAD_ACCUM_STEPS),
    "--learning_rate", str(LEARNING_RATE),
    "--use_pro_version", "True",
    "--use_gradient_checkpointing", "True",
    "--balance_z_loss", "False",
    # Left at its 0.1 default, the warmup block rewrites the learning rate back
    # to --learning_rate on every step, which silently cancels the 10x decay at
    # --num_steps_before_decay. 0 disables it and leaves the scheduler in charge.
    "--lr_warmup_steps", "0",
    "--wandb_project", WANDB_PROJECT,
]
# Only when set: draccus reads an empty value as the string "None".
if WANDB_ENTITY:
    command += ["--wandb_entity", WANDB_ENTITY]
if RESUME_DIR:
    command += ["--resume_checkpoint", str(RESUME_DIR)]
    if RESUME_LEARNING_RATE is not None:
        command += ["--resume_learning_rate", str(RESUME_LEARNING_RATE)]
train_env = os.environ.copy()
train_env.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "WANDB_MODE": WANDB_MODE,
    "WANDB_RUN_ID": RUN_ID,
    "WANDB_RESUME": "allow",
    "PYTHONPATH": str(VLA_DIR),
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TF_CPP_MIN_LOG_LEVEL": "2",
})
run(command, cwd=VLA_DIR, env=train_env)


## 7. Package the fully fine-tuned model

This archive contains the full VLA weights and config, action head, proprio projector, processor files, action-normalization statistics, and optimizer/scheduler state for resuming. The full model includes the trained vision and language backbones and action queries; no separate base-model weights or LoRA merge are needed. Full checkpoints and their archives are substantially larger than LoRA adapters. The repository's current `rollout.py` loader still expects LoRA artifacts and will need full-checkpoint loading support before it can run this model.

For an unattended overnight run, leave `SAVE_TO_DRIVE = True` and set `SHUTDOWN_WHEN_DONE = True`: the archive is copied to Drive and the last cell releases the runtime instead of letting it idle until Colab reclaims it. Use **Runtime -> Run all** and keep the browser tab open unless your Colab plan includes background execution.

In [ ]:
import tarfile

RUN_DIR = OUTPUT_ROOT / RUN_ID
require_full_checkpoint(RUN_DIR)
ARCHIVE = ROOT / f"{RUN_ID}.tar.gz"
with tarfile.open(ARCHIVE, "w:gz") as archive:
    archive.add(RUN_DIR, arcname=RUN_ID)
print(f"Created {ARCHIVE} ({ARCHIVE.stat().st_size / 2**20:.1f} MiB)")

if SAVE_TO_DRIVE:
    destination = Path(DRIVE_OUTPUT) / ARCHIVE.name
    shutil.copy2(ARCHIVE, destination)
    print("Copied to", destination)
else:
    from google.colab import files
    files.download(str(ARCHIVE))


In [ ]:
# Release the runtime, so an overnight run stops billing when it finishes
# rather than when Colab's idle timeout notices. Everything on the local disk
# goes with it -- this cell is deliberately last, after the export above.
#
# A failing cell leaves the runtime up: Colab abandons the rest of a queued
# run on the first error, so this never fires and the traceback survives for
# as long as the idle timeout allows.
if SHUTDOWN_WHEN_DONE:
    if not SAVE_TO_DRIVE:
        raise RuntimeError(
            "SAVE_TO_DRIVE is False: checkpoints live on the runtime disk and "
            "would be destroyed. Save them before shutting down."
        )
    print("Archive at", destination)
    from google.colab import runtime
    runtime.unassign()
